#### FFW SG2 MetaQuest3 real render

관절 상태(5560)는 현재 joint/EEF/VR과 같은 strict JSON worker에서 받습니다. 현재 [worker 코드](../../../../VR_teleoperation/package/worker/)를 사용하세요. 렌더 목표(5556)는 기존 pickle 메시지를 유지합니다.


In [7]:
# Locate the shared workspace from a notebook, project or repository directory.
from pathlib import Path
import runpy

_notebook_dir = (Path(__vsc_ipynb_file__).resolve().parent
                 if "__vsc_ipynb_file__" in globals() else Path.cwd())
_search_bases = [_notebook_dir, *_notebook_dir.parents, Path.cwd(), *Path.cwd().parents]
_candidates = [candidate for base in _search_bases
               for candidate in (base, base / "VR_teleoperation")]
_root = next((candidate for candidate in _candidates
              if (candidate / ".project_v2").is_file()), None)
if _root is None:
    raise RuntimeError("Open this notebook from the VR_teleoperation workspace.")
_bootstrap = runpy.run_path(str(_root / "package" / "init_project.py"))
for _name in _bootstrap["__all__"]:
    globals()[_name] = _bootstrap[_name]

import numpy as np

from ri_motion_v5_package.init_scripts.init_ipython_setup import *
from ri_motion_v5_package.init_scripts.init_qt import *
from ri_motion_v5_package.mujoco_sim import *
from ri_motion_v5_package.kinematics import *
from ri_motion_v5_package.utility import *
from ri_motion_v5_package.qt import *
from utils import *

app = get_qt_app()

from package.robot_client import SG2RobotStateSubscriber



[ffw_sg2_vr_teleoperation] initialized.
 - PROJECT_DIR: /Users/jaehoon/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation


In [8]:
RENDER_PORT   = 5556
RENDER_PUB_IP = "127.0.0.1"
SG2_SUB_PORT  = 5560
ROBOT_SUB_IP  = "192.168.6.2"

hand_types = list(SG2_HAND_TYPES)
hand_labels = dict(SG2_HAND_LABELS)

print("Render SUB:","tcp://%s:%d"%(RENDER_PUB_IP,RENDER_PORT))
print("Robot SUB :","tcp://%s:%d"%(ROBOT_SUB_IP,SG2_SUB_PORT))


Render SUB: tcp://127.0.0.1:5556
Robot SUB : tcp://192.168.6.2:5560


In [9]:
SG2_RENDER_SCENE_XML_PATH.parent.mkdir(parents=True,exist_ok=True)
xml_path = merge_mjcfs(
    included_mjcf_files = [
        str(FLOOR_XML_PATH),
        str(SG2_XML_PATH),
        str(SG2_BODY_ADDED_VIS_XML_PATH),
    ],
    output_xml_path = str(SG2_RENDER_SCENE_XML_PATH),)
env = MuJoCoParser(rel_xml_path=xml_path,verbose=False)
env.reset()
env.set_p("base_link","body",(0.0,0.0,0.01))
env.set_p("vis_base_link","body",(0.0,0.0,0.01))


[merge_mjcfs] Merging [3] MJCF files:
  - [0] [/Users/jaehoon/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/floor_simple_white.xml]
  - [1] [/Users/jaehoon/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/ffw_sg2.xml]
  - [2] [/Users/jaehoon/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/ffw_sg2_body_added_vis.xml]
[merge_mjcfs] Saved merged XML to:[/Users/jaehoon/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/.cache/scene_ffw_sg2_metaquest3_render.xml]
MuJoCo model loaded from [/Users/jaehoon/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/.cache/scene_ffw_sg2_metaquest3_render.xml].


In [10]:
real_command_joint_names = (
    list(SG2_REAL_COMMAND_JOINT_NAMES)
    + [SG2_GRIPPER_DRIVER_JOINT_NAMES[hand_type] for hand_type in hand_types]
)
command_joint_names = filter_existing_joint_names(env,real_command_joint_names)

q_msg_format_dict = {
    "lift"      : {"prefixes":["lift_joint"]},
    "head"      : {"prefixes":["head_joint"]},
    "arm_l"     : {"prefixes":["arm_l_joint"]},
    "arm_r"     : {"prefixes":["arm_r_joint"]},
    "gripper_l" : {"prefixes":["gripper_l_joint1"]},
    "gripper_r" : {"prefixes":["gripper_r_joint1"]},
}
q_msg_layout = build_q_msg_layout(
    env               = env,
    q_msg_format_dict = q_msg_format_dict,
    use_joint_names   = command_joint_names,)

q_vis_msg_format_dict = {
    "lift"      : {"prefixes":["vis_lift_joint"]},
    "head"      : {"prefixes":["vis_head_joint"]},
    "arm_l"     : {"prefixes":["vis_arm_l_joint"]},
    "arm_r"     : {"prefixes":["vis_arm_r_joint"]},
    "gripper_l" : {"prefixes":["vis_gripper_l_joint"]},
    "gripper_r" : {"prefixes":["vis_gripper_r_joint"]},
}
q_vis_msg_layout = build_q_msg_layout(
    env               = env,
    q_msg_format_dict = q_vis_msg_format_dict,)

real_joint_names = q_msg_layout["use_joint_names"]
vis_joint_names = q_vis_msg_layout["use_joint_names"]
vis_joint_set = set(vis_joint_names)

q_init = env.get_qpos(joint_names=command_joint_names)
env.forward(q=q_init,joint_names=command_joint_names)
vis_names_init,vis_qpos_init = get_sg2_vis_q_from_render_msg(
    {"joint_names":command_joint_names,"qpos":q_init},
    vis_joint_set,)
if len(vis_names_init) > 0:
    env.forward(q=vis_qpos_init,joint_names=vis_names_init)

print("SG2 render joint layout")
print("-----------------------")
for q_msg_key in q_msg_layout["q_msg_keys"]:
    real_names = q_msg_layout["qmsgkeys_to_usejointnames_dict"][q_msg_key]
    vis_names = q_vis_msg_layout["qmsgkeys_to_usejointnames_dict"][q_msg_key]
    print("%-9s | real:%2d | vis:%2d"%(q_msg_key,len(real_names),len(vis_names)))
print("-----------------------")
print("%-9s | real:%2d | vis:%2d"%("total",len(real_joint_names),len(vis_joint_names)))


SG2 render joint layout
-----------------------
lift      | real: 1 | vis: 1
head      | real: 2 | vis: 2
arm_l     | real: 7 | vis: 7
arm_r     | real: 7 | vis: 7
gripper_l | real: 1 | vis: 4
gripper_r | real: 1 | vis: 4
-----------------------
total     | real:19 | vis:25


In [11]:
sub_render = ZMQPubSub(role="sub",ip=RENDER_PUB_IP,port=RENDER_PORT,verbose=True,)
sub_robot = SG2RobotStateSubscriber(robot_ip=ROBOT_SUB_IP, state_port=SG2_SUB_PORT)

print("ZMQ subscribers")
print("---------------")
print("Render pub  : tcp://%s:%d"%(RENDER_PUB_IP,RENDER_PORT))
print("Robot state : tcp://%s:%d (strict JSON)"%(ROBOT_SUB_IP,SG2_SUB_PORT))


[ZMQ Subscriber] pid:[41554] endpoint:[tcp://127.0.0.1:5556]
[ZMQ] role:[sub] ip:[127.0.0.1] port:[5556] hwm:[1] conflate:[True]
[ZMQ Subscriber] pid:[41554] endpoint:[tcp://192.168.6.2:5560]
[ZMQ] role:[sub] ip:[192.168.6.2] port:[5560] hwm:[1] conflate:[True]
ZMQ subscribers
---------------
Render pub  : tcp://127.0.0.1:5556
Robot state : tcp://192.168.6.2:5560


In [12]:
env.init_viewer(title="FFW SG2 MetaQuest3 Render",x_offset=0.24,y_offset=0.00,width=0.76,height=1.0,fontscale=180)
env.viewer.set_cam_info(180.0,2.8,-20.0,[0.0,0.0,0.95])
env.viewer.set_transparency(transparent=True)
env.viewer.set_geomgroup(group_2=True,group_3=False,group_4=True)
env.viewer.set_sitegroup(group_0=False)

render_hz = 20
tmr_render = SimpleTimer(name="SG2 Render",Hz=render_hz,verbose=True)
tmr_render.start(); env.reset_wall_time(); env.reset_sim_time()

pub_robot_flag = False
last_render_msg = {}


def _as_T(T):
    if T is None:
        return None
    try:
        return np.asarray(T,dtype=np.float64).reshape(4,4)
    except Exception:
        return None


def _as_p(p):
    if p is None:
        return None
    try:
        return np.asarray(p,dtype=np.float64).reshape(3)
    except Exception:
        return None


def _overlay_onoff(v):
    return "ON" if bool(v) else "OFF"


def _overlay_float(v,digits=4):
    if v is None:
        return "-"
    try:
        v = float(v)
        if not np.isfinite(v):
            return "-"
        return ("%%.%df"%digits)%v
    except (TypeError,ValueError):
        return str(v)


def _overlay_metric(v):
    if isinstance(v,dict):
        return "L:%s R:%s"%(_overlay_float(v.get("left",None)),_overlay_float(v.get("right",None)))
    return _overlay_float(v)


def _overlay_phase(d,key):
    if not isinstance(d,dict):
        return "-"
    v = d.get(key,"-")
    return "-" if v is None else str(v)


def _overlay_progress(d,key):
    if not isinstance(d,dict):
        return "-"
    v = d.get(key,None)
    if v is None:
        return "-"
    return "%.0f%%"%(100.0*float(v),)


def _sort_payload_key(key):
    try:
        return (0,int(key))
    except Exception:
        return (1,str(key))


def _get_hand_payload(msg,key,hand_type):
    payload = msg.get(key,None) if isinstance(msg,dict) else None
    if not isinstance(payload,dict):
        return None
    return payload.get(hand_type,None)


def _plot_T_payload(T,axis_len,axis_width,axis_alpha,label):
    T = _as_T(T)
    if T is None:
        return
    env.plot_T(T=T,axis_len=axis_len,axis_width=axis_width,axis_alpha=axis_alpha,label=label,)


def _plot_quest_nodes(msg):
    if not isinstance(msg,dict) or not msg.get("quest_all_ok",False):
        return
    raw_nodes = msg.get("T_quest_all_by_idx",{})
    virtual_nodes = msg.get("T_quest_virtual_by_idx",{})
    if not isinstance(raw_nodes,dict): raw_nodes = {}
    if not isinstance(virtual_nodes,dict): virtual_nodes = {}
    nodes_to_plot = dict(raw_nodes)
    nodes_to_plot.update(virtual_nodes)
    for plot_idx,key in enumerate(sorted(nodes_to_plot.keys(),key=_sort_payload_key)):
        T_node = _as_T(nodes_to_plot[key])
        if T_node is None:
            continue
        env.plot_T(T=T_node,axis_len=0.035,axis_width=0.002,axis_alpha=0.35,label="Quest virtual nodes" if plot_idx == 0 else "",)


def _plot_ik_targets(targets,r,rgba,label=""):
    if not isinstance(targets,(list,tuple)):
        return
    for idx,target in enumerate(targets):
        if not isinstance(target,dict):
            continue
        p_curr = _as_p(target.get("p_curr",None))
        p_trgt = _as_p(target.get("p_trgt",None))
        if p_curr is None or p_trgt is None:
            continue
        env.plot_sphere(p_curr,r,rgba=rgba)
        env.plot_sphere(p_trgt,r,rgba=rgba,label=label if idx == 0 else "")
        env.plot_arrow_fr2to(p_fr=p_curr,p_to=p_trgt,rgba=rgba,r=r*0.5)


def _plot_metaquest_payload(msg):
    if not isinstance(msg,dict):
        return
    _plot_T_payload(msg.get("T_quest_start",None),0.12,0.005,0.45,"Quest T_start")
    _plot_quest_nodes(msg)
    for hand_type in hand_types:
        label = hand_labels[hand_type]
        _plot_T_payload(_get_hand_payload(msg,"T_quest_wrist",hand_type),0.13,0.006,0.90,"%s Quest wrist"%label)
        _plot_T_payload(_get_hand_payload(msg,"T_quest_virtual_wrist",hand_type),0.10,0.004,0.55,"%s virtual wrist"%label)
        _plot_T_payload(_get_hand_payload(msg,"T_quest_target_raw",hand_type),0.11,0.005,0.80,"%s raw target"%label)
        _plot_T_payload(_get_hand_payload(msg,"T_quest_target",hand_type),0.10,0.004,0.60,"%s attach target"%label)
        _plot_T_payload(_get_hand_payload(msg,"T_sg2_wrist",hand_type),0.11,0.005,0.70,"%s SG2 wrist"%label)
    _plot_ik_targets(msg.get("active_arm_ik_targets",[]),r=0.006,rgba=(0.0,0.4,1.0,0.6),label="arm IK targets",)


def _draw_overlays(msg):
    raw_cmd_flag = msg.get("raw_robot_command_on",pub_robot_flag) if isinstance(msg,dict) else pub_robot_flag
    env.viewer_text_overlay("Command","raw %s | pub %s"%(_overlay_onoff(raw_cmd_flag),_overlay_onoff(pub_robot_flag)),loc="top left")
    env.viewer_text_overlay("Orange Viz","command qpos",loc="top left")
    if not isinstance(msg,dict):
        return
    quest_ok = msg.get("quest_ok",{})
    quest_text = "All %s"%("OK" if msg.get("quest_all_ok",False) else "MISS",)
    if isinstance(quest_ok,dict):
        quest_text += " | " + " | ".join(["%s %s"%(hand_labels[hand_type],"OK" if quest_ok.get(hand_type,False) else "MISS") for hand_type in hand_types])
    env.viewer_text_overlay("Quest",quest_text,loc="top left")

    attached = msg.get("attached",{})
    attach_phase = msg.get("attach_phase",{})
    attach_progress = msg.get("attach_progress",{})
    attach_text = " | ".join(["%s %s/%s %s"%(hand_labels[hand_type],_overlay_onoff(attached.get(hand_type,False)) if isinstance(attached,dict) else "OFF",_overlay_phase(attach_phase,hand_type),_overlay_progress(attach_progress,hand_type),) for hand_type in hand_types])
    env.viewer_text_overlay("Attach",attach_text,loc="top left")
    env.viewer_text_overlay("Gesture",make_sg2_detach_gesture_overlay_text(hand_types,hand_labels,msg.get("detach_gesture_state",None)),loc="top left")

    env.viewer_text_overlay("IK","arm %s | mode %s | min dist %s"%(_overlay_metric(msg.get("arm_ik_err_best",None)),str(msg.get("collision_ik_mode","-")),_overlay_float(msg.get("min_signed_dist",None)),),loc="top left",)
    env.viewer_text_overlay("Rate","robot %.0fHz/%sHz | render %.0fHz"%(float(msg.get("robot_hz",0.0)),_overlay_float(msg.get("robot_pub_hz_actual",0.0),digits=1),float(msg.get("render_hz",0.0)),),loc="top left",)


try:
    while env.is_viewer_alive():
        app.processEvents()

        render_msg = sub_render.recv()
        if render_msg is not None:
            last_render_msg = render_msg
            pub_robot_flag = bool(render_msg.get("pub_robot",False))
            if "joint_names" in render_msg and "qpos" in render_msg:
                vis_names,vis_qpos = get_sg2_vis_q_from_render_msg(render_msg,vis_joint_set)
                if len(vis_names) > 0:
                    env.forward(q=vis_qpos,joint_names=vis_names)

        robot_state = sub_robot.recv()
        if robot_state is not None:
            real_names,real_qpos = get_real_q_from_robot_state(
                robot_state  = robot_state,
                q_msg_layout = q_msg_layout,)
            if len(real_names) > 0:
                env.forward(q=real_qpos,joint_names=real_names)

        if sub_render.is_pub_closed():
            print("Render publisher closed.")
            break

        if tmr_render.do_run():
            env.plot_global_coordinate_axes()
            env.plot_links_between_bodies(r=0.001)
            _plot_metaquest_payload(last_render_msg)
            _draw_overlays(last_render_msg)
            env.render()
            tmr_render.end()
except KeyboardInterrupt:
    print("Interrupted.")
finally:
    if getattr(env,"final_rgb_img",None) is not None:
        imshow(env.final_rgb_img,fig_w=6,title="Final scene",title_fs=8)
    sub_render.close()
    sub_robot.close()
    env.close_viewer()
    print("Done.")


[SG2 Render] Initialized with [20.0]HZ. MAX_SEC:[inf]sec.
[SG2 Render] Start ([20.0]HZ. MAX_SEC:[inf]sec).
[SG2 Render][1] run delayed! T:[50.0]ms run:[134.8]ms]
[SG2 Render][2][0.1s] delayed! T:[50.0]ms interval:[135.4]ms prev_run:[134.8]ms.
[SG2 Render][1016][50.8s] delayed! T:[50.0]ms interval:[56.4]ms prev_run:[24.7]ms.
[SG2 Render][1713] run delayed! T:[50.0]ms run:[50.8]ms]
[SG2 Render][1917] run delayed! T:[50.0]ms run:[51.4]ms]
[SG2 Render][1952] run delayed! T:[50.0]ms run:[51.5]ms]
[SG2 Render][2804][140.2s] delayed! T:[50.0]ms interval:[118.7]ms prev_run:[29.8]ms.
[ZMQ SUB] pid:[41554] Socket closed.
[ZMQ SUB] pid:[41554] Socket closed.
Done.
